In [5]:
# FINAL POST-AUDIT GOLD -> v2.2 VALIDATION
#
# Uses only Gold rows marked Done.
# Paper matching: canonical ACL URL.
# High-confidence dataset alignment: exact normalized dataset name;
# if no name match exists, exact non-empty dataset URL within the same paper.
#
# Geography is evidence-matched:
#   Gold Explicit       -> v2.2 explicit country set
#   Gold Explicit+Inferred -> v2.2 explicit+inferred country set
#
# Important: rates over all Gold references and rates conditional on v2.2
# making a geographic prediction are both reported so abstention is visible.

from pathlib import Path
import ast
import re
import numpy as np
import os 
import pandas as pd


DATA_DIR = Path(os.environ.get("ATLASNLP_DATA_DIR", ".")).expanduser().resolve()
GOLD_VALIDATION_DIR = DATA_DIR / "post_audit_gold_validation"
GOLD_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)



gold_path = resolve_file(
    "human_worldbench_post_machine_assisted_audit.csv",
    extra_patterns=["human_worldbench_post_machine_assisted_audit*.csv"],
)
gold_post = read_csv_flexible(gold_path)

def canonical_url(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"[?#].*$", "", s)
    s = s.rstrip("/")
    s = re.sub(r"\.pdf$", "", s)
    return s.rstrip("/")

def canonical_dataset_url(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"[?#].*$", "", s)
    return s.rstrip("/")

def normalize_dataset_name(x):
    if pd.isna(x):
        return ""
    s = str(x).lower().strip()
    s = re.sub(r"[\W_]+", " ", s, flags=re.UNICODE)
    return " ".join(s.split())

def normalize_task_key(x):
    if pd.isna(x):
        return ""
    s = str(x).lower().replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return " ".join(s.split())

GOLD_COUNTRY_ALIASES = {
    # Match the final v2.2 ontology / normalization policy.
    "Hong Kong": "China",
}

def parse_gold_country_field(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "not stated"}:
        return []
    # Gold multi-country lists are comma-separated.
    return [p.strip() for p in s.split(",") if p.strip()]

def gold_row_country_set(row):
    values = []
    if pd.notna(row["Assigned country"]):
        values.append(str(row["Assigned country"]).strip())
    values.extend(
        parse_gold_country_field(
            row["All Countries Covered (you can select multiple countries)"]
        )
    )
    return {
        GOLD_COUNTRY_ALIASES.get(v, v)
        for v in values
        if v
    }

def parse_v2_country_set(x):
    if isinstance(x, (list, tuple, set)):
        return set(x)
    if pd.isna(x):
        return set()
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "[]"}:
        return set()
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple, set)):
            return {str(v).strip() for v in obj if str(v).strip()}
    except Exception:
        pass
    return {
        p.strip()
        for p in re.split(r";|\|", s)
        if p.strip()
    }

def gold_evidence_class(x):
    s = str(x).strip().lower()
    if s.startswith("explicit"):
        return "Explicit"
    if s.startswith("inferred"):
        return "Inferred"
    return "Other"

# ---------------------------------------------------------------------
# 1. Freeze the post-audit Gold reference population.
# ---------------------------------------------------------------------
gold_done = gold_post[
    gold_post["Validation Status (reviewers only)"].eq("Done")
].copy()

gold_done["paper_key"] = gold_done["Paper Link (URL)"].apply(canonical_url)
gold_done["gold_name_key"] = gold_done["Dataset name"].apply(normalize_dataset_name)
gold_done["gold_dataset_url_key"] = gold_done["Dataset Link (URL)"].apply(
    canonical_dataset_url
)
gold_done["gold_evidence_class"] = gold_done[
    "Country Attribution Method"
].apply(gold_evidence_class)

gold_acl = gold_done[
    gold_done["paper_key"].str.contains("aclanthology.org", na=False)
].copy()

# Normalize Core matching fields independently of legacy paper_url_key casing.
core_eval = core_all.copy()
core_eval["paper_key_eval"] = core_eval["paper_url"].apply(canonical_url)
core_eval["core_name_key"] = core_eval["Dataset name"].apply(normalize_dataset_name)
core_eval["core_dataset_url_key"] = core_eval["Dataset Link (URL)"].apply(
    canonical_dataset_url
)

gold_acl_papers = set(gold_acl["paper_key"])
initial_recovered = core_eval[
    core_eval["paper_key_eval"].isin(gold_acl_papers)
].copy()
final_eligible_recovered = initial_recovered[
    initial_recovered["v2_primary_eligible"].eq(True)
].copy()
excluded_recovered = initial_recovered[
    initial_recovered["v2_primary_eligible"].eq(False)
].copy()

scope_summary = pd.DataFrame({
    "stage": [
        "Post-audit Gold rows marked Done",
        "ACL-linked Gold papers",
        "Recovered in initial 18,035 extraction set",
        "Retained in final 13,462 Core",
        "Excluded after post-extraction role audit",
    ],
    "n": [
        len(gold_done),
        gold_acl["paper_key"].nunique(),
        initial_recovered["paper_key_eval"].nunique(),
        final_eligible_recovered["paper_key_eval"].nunique(),
        excluded_recovered["paper_key_eval"].nunique(),
    ],
})
display(scope_summary)

print("\nWhy recovered Gold papers were excluded from final Core:")
display(
    excluded_recovered["v2_dataset_relationship"]
    .value_counts(dropna=False)
    .rename("papers")
    .to_frame()
)

# ---------------------------------------------------------------------
# 2. High-confidence dataset-level alignment within recovered eligible papers.
# ---------------------------------------------------------------------
core_tasks = sorted(
    core_all["normalized_task_category"].dropna().astype(str).unique()
)
core_task_map = {normalize_task_key(t): t for t in core_tasks}

validation_rows = []

for _, core_row in final_eligible_recovered.iterrows():
    paper_key = core_row["paper_key_eval"]
    gold_paper = gold_acl[gold_acl["paper_key"].eq(paper_key)].copy()

    core_name = core_row["core_name_key"]
    core_dataset_url = core_row["core_dataset_url_key"]

    # Prefer exact normalized dataset-name alignment.
    name_match = gold_paper["gold_name_key"].eq(core_name)

    if name_match.any():
        matched_gold = gold_paper[name_match].copy()
        match_type = "exact normalized dataset name"

    # If the names differ, accept an exact non-empty dataset URL within the same paper.
    elif core_dataset_url:
        url_match = (
            gold_paper["gold_dataset_url_key"].ne("")
            & gold_paper["gold_dataset_url_key"].eq(core_dataset_url)
        )
        if not url_match.any():
            continue
        matched_gold = gold_paper[url_match].copy()
        match_type = "exact dataset URL"

    else:
        continue

    # Prevent accidental aggregation of two different Gold targets.
    # Under the audited file this should be one normalized Gold dataset-name group.
    if matched_gold["gold_name_key"].nunique() != 1:
        continue

    gold_explicit = set()
    gold_any = set()

    for _, gold_row in matched_gold.iterrows():
        countries = gold_row_country_set(gold_row)
        method = gold_row["gold_evidence_class"]

        if method == "Explicit":
            gold_explicit |= countries
            gold_any |= countries
        elif method == "Inferred":
            gold_any |= countries

    gold_task_set = set()
    unmatched_gold_tasks = set()

    for task in matched_gold["Task Category"].dropna().astype(str).unique():
        key = normalize_task_key(task)
        if key in core_task_map:
            gold_task_set.add(core_task_map[key])
        else:
            unmatched_gold_tasks.add(task)

    validation_rows.append({
        "audit_record_id": core_row["audit_record_id"],
        "paper_key": paper_key,
        "match_type": match_type,
        "gold_dataset_name": matched_gold["Dataset name"].iloc[0],
        "core_dataset_name": core_row["Dataset name"],
        "gold_explicit_set": gold_explicit,
        "gold_any_set": gold_any,
        "v2_explicit_set": parse_v2_country_set(
            core_row["explicit_content_countries_normalized"]
        ),
        "v2_any_set": parse_v2_country_set(
            core_row["explicit_plus_inferred_countries_normalized"]
        ),
        "gold_task_set": gold_task_set,
        "gold_unmatched_tasks": unmatched_gold_tasks,
        "v2_task": core_row["normalized_task_category"],
        "v2_country_status": core_row["content_country_status_normalized"],
        "v2_dataset_relationship": core_row["v2_dataset_relationship"],
    })

gold_v2 = pd.DataFrame(validation_rows)

print("\nHigh-confidence dataset-level matches:")
display(
    gold_v2["match_type"]
    .value_counts()
    .rename("records")
    .to_frame()
)

# ---------------------------------------------------------------------
# 3. Geography metrics.
# ---------------------------------------------------------------------
def country_validation_metrics(df, pred_col, ref_col):
    pairs = list(zip(df[pred_col], df[ref_col]))
    predicted = [(p, r) for p, r in pairs if p]

    tp = sum(len(p & r) for p, r in pairs)
    pred_assoc = sum(len(p) for p, _ in pairs)
    ref_assoc = sum(len(r) for _, r in pairs)

    n = len(df)
    n_pred = len(predicted)

    return pd.Series({
        "gold_reference_records": n,
        "v2_records_with_country": n_pred,
        "v2_country_coverage": n_pred / n if n else np.nan,
        "any_overlap_all_gold_records": (
            sum(bool(p & r) for p, r in pairs) / n if n else np.nan
        ),
        "exact_set_all_gold_records": (
            sum(p == r for p, r in pairs) / n if n else np.nan
        ),
        "overlap_given_v2_prediction": (
            sum(bool(p & r) for p, r in predicted) / n_pred
            if n_pred else np.nan
        ),
        "exact_set_given_v2_prediction": (
            sum(p == r for p, r in predicted) / n_pred
            if n_pred else np.nan
        ),
        "micro_country_precision": (
            tp / pred_assoc if pred_assoc else np.nan
        ),
        "micro_country_recall": (
            tp / ref_assoc if ref_assoc else np.nan
        ),
        "v2_country_associations": pred_assoc,
        "gold_country_associations": ref_assoc,
        "overlapping_country_associations": tp,
    })

gold_explicit_eval = gold_v2[
    gold_v2["gold_explicit_set"].map(bool)
].copy()

gold_sensitivity_eval = gold_v2[
    gold_v2["gold_any_set"].map(bool)
].copy()

gold_inferred_only_eval = gold_v2[
    ~gold_v2["gold_explicit_set"].map(bool)
    & gold_v2["gold_any_set"].map(bool)
].copy()

country_summary = pd.DataFrame({
    "Gold explicit -> v2 explicit": country_validation_metrics(
        gold_explicit_eval,
        "v2_explicit_set",
        "gold_explicit_set",
    ),
    "Gold explicit+inferred -> v2 explicit+inferred": country_validation_metrics(
        gold_sensitivity_eval,
        "v2_any_set",
        "gold_any_set",
    ),
    "Gold inferred-only -> v2 explicit+inferred": country_validation_metrics(
        gold_inferred_only_eval,
        "v2_any_set",
        "gold_any_set",
    ),
}).T

display(
    country_summary.style.format({
        "v2_country_coverage": "{:.1%}",
        "any_overlap_all_gold_records": "{:.1%}",
        "exact_set_all_gold_records": "{:.1%}",
        "overlap_given_v2_prediction": "{:.1%}",
        "exact_set_given_v2_prediction": "{:.1%}",
        "micro_country_precision": "{:.1%}",
        "micro_country_recall": "{:.1%}",
    })
)

# ---------------------------------------------------------------------
# 4. Single- vs multi-country structure.
# ---------------------------------------------------------------------
country_structure_rows = []

for label, frame, pred_col, ref_col in [
    (
        "Explicit / single-country Gold reference",
        gold_explicit_eval[
            gold_explicit_eval["gold_explicit_set"].map(len).eq(1)
        ],
        "v2_explicit_set",
        "gold_explicit_set",
    ),
    (
        "Explicit / multi-country Gold reference",
        gold_explicit_eval[
            gold_explicit_eval["gold_explicit_set"].map(len).gt(1)
        ],
        "v2_explicit_set",
        "gold_explicit_set",
    ),
    (
        "Sensitivity / single-country Gold reference",
        gold_sensitivity_eval[
            gold_sensitivity_eval["gold_any_set"].map(len).eq(1)
        ],
        "v2_any_set",
        "gold_any_set",
    ),
    (
        "Sensitivity / multi-country Gold reference",
        gold_sensitivity_eval[
            gold_sensitivity_eval["gold_any_set"].map(len).gt(1)
        ],
        "v2_any_set",
        "gold_any_set",
    ),
]:
    row = country_validation_metrics(frame, pred_col, ref_col)
    row["group"] = label
    country_structure_rows.append(row)

country_structure = pd.DataFrame(country_structure_rows).set_index("group")

print("\nSingle- vs multi-country validation:")
display(
    country_structure.style.format({
        "v2_country_coverage": "{:.1%}",
        "any_overlap_all_gold_records": "{:.1%}",
        "exact_set_all_gold_records": "{:.1%}",
        "overlap_given_v2_prediction": "{:.1%}",
        "exact_set_given_v2_prediction": "{:.1%}",
        "micro_country_precision": "{:.1%}",
        "micro_country_recall": "{:.1%}",
    })
)

# ---------------------------------------------------------------------
# 5. Task validation.
# ---------------------------------------------------------------------
task_eval = gold_v2[
    gold_v2["gold_task_set"].map(len).eq(1)
    & gold_v2["gold_unmatched_tasks"].map(len).eq(0)
].copy()

task_eval["gold_task"] = task_eval["gold_task_set"].apply(
    lambda s: next(iter(s))
)
task_eval["task_exact_match"] = task_eval["gold_task"].eq(
    task_eval["v2_task"]
)

task_summary = pd.DataFrame({
    "metric": [
        "High-confidence aligned records",
        "Records with one comparable Gold task label",
        "Exact task matches",
        "Exact task agreement",
    ],
    "value": [
        len(gold_v2),
        len(task_eval),
        int(task_eval["task_exact_match"].sum()),
        float(task_eval["task_exact_match"].mean()),
    ],
})
display(task_summary)

# ---------------------------------------------------------------------
# 6. Inspect the small number of positive geographic disagreements.
# ---------------------------------------------------------------------
gold_v2["explicit_overlap"] = gold_v2.apply(
    lambda r: bool(r["gold_explicit_set"] & r["v2_explicit_set"]),
    axis=1,
)
gold_v2["sensitivity_overlap"] = gold_v2.apply(
    lambda r: bool(r["gold_any_set"] & r["v2_any_set"]),
    axis=1,
)

positive_disagreements = gold_v2[
    (
        gold_v2["v2_explicit_set"].map(bool)
        & gold_v2["gold_explicit_set"].map(bool)
        & ~gold_v2["explicit_overlap"]
    )
    |
    (
        gold_v2["v2_any_set"].map(bool)
        & gold_v2["gold_any_set"].map(bool)
        & ~gold_v2["sensitivity_overlap"]
    )
].copy()

print("\nPositive geographic disagreements to inspect:")
display(
    positive_disagreements[
        [
            "paper_key",
            "gold_dataset_name",
            "core_dataset_name",
            "gold_explicit_set",
            "v2_explicit_set",
            "gold_any_set",
            "v2_any_set",
            "v2_country_status",
        ]
    ]
)

# ---------------------------------------------------------------------
# 7. Export reproducibility artifacts.
# ---------------------------------------------------------------------
def set_as_text(x):
    return "; ".join(sorted(x)) if isinstance(x, set) else x

gold_v2_export = gold_v2.copy()
for col in [
    "gold_explicit_set",
    "gold_any_set",
    "v2_explicit_set",
    "v2_any_set",
    "gold_task_set",
    "gold_unmatched_tasks",
]:
    gold_v2_export[col] = gold_v2_export[col].apply(set_as_text)

scope_summary.to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_scope_summary.csv",
    index=False,
)
country_summary.to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_country_validation_summary.csv"
)
country_structure.to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_country_structure_summary.csv"
)
task_summary.to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_task_validation_summary.csv",
    index=False,
)
gold_v2_export.to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_high_confidence_matches.csv",
    index=False,
)
positive_disagreements[
    [
        "paper_key",
        "gold_dataset_name",
        "core_dataset_name",
        "gold_explicit_set",
        "v2_explicit_set",
        "gold_any_set",
        "v2_any_set",
        "v2_country_status",
    ]
].assign(
    gold_explicit_set=lambda d: d["gold_explicit_set"].apply(set_as_text),
    v2_explicit_set=lambda d: d["v2_explicit_set"].apply(set_as_text),
    gold_any_set=lambda d: d["gold_any_set"].apply(set_as_text),
    v2_any_set=lambda d: d["v2_any_set"].apply(set_as_text),
).to_csv(
    GOLD_VALIDATION_DIR / "gold_v2_positive_geography_disagreements.csv",
    index=False,
)

print(f"\nValidation artifacts written to: {GOLD_VALIDATION_DIR}")


NameError: name 'core_all' is not defined